# Natywne baseline’y: LeWM i LpWM na PushT

Najpierw sprawdzamy opublikowany checkpoint LeWM w jego własnym repo i symulatorze.
Następnie trenujemy LeWM przez 10 epok zgodnie z publikacją oraz, osobno, LpWM przez 10
z oceną checkpointów 2/5/10. To osobne protokoły: ich success rate nie jest jeszcze
wspólnym benchmarkiem. Nasze modele sparse-generator pozostają osobną ablacją.

PHASE wybiera jeden etap; domyślnie przygotowanie. Przygotowanie danych można wykonać na CPU;
planowanie/trening wymagają GPU. Wpisz **aktualne pozostałe jednostki**, nie pierwotne 1800.
Pobieranie, instalacja i bezczynność GPU nie są ujęte w liczniku komend.


In [ ]:
import os, sys, json, subprocess
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
REPO = Path('/content/lpwm-native-launcher')
REPO_REF = 'experiment/theory-colab-1800'
if not REPO.exists():
    subprocess.run(['git','clone','--branch',REPO_REF,'--single-branch',
                    'https://github.com/twojtys137/lpworldmodel.git',str(REPO)],check=True)
if subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip():
    raise RuntimeError('Repo ma lokalne zmiany; wybierz nowy katalog lub zachowaj je przed aktualizacją.')
subprocess.run(['git','-C',str(REPO),'fetch','origin',REPO_REF],check=True)
COMMIT = subprocess.check_output(['git','-C',str(REPO),'rev-parse','FETCH_HEAD'],text=True).strip()
subprocess.run(['git','-C',str(REPO),'checkout','--detach',COMMIT],check=True)
os.chdir(REPO)
print('Launcher commit:', COMMIT)

WORK = Path('/content/native-worldmodels')
DATA = Path('/content/native-wm-data')
OUTPUT = Path('/content/drive/MyDrive/lpwm-native-v1')
REMAINING_CU = None  # wpisz aktualny pozostały budżet, np.1200
RATE_CU_HOUR = None  # wpisz obecną stawkę z panelu zasobów Colaba
PHASE = 'setup'  # setup | published_eval | lewm_train | lpwm_train
RUN_NAME = 'lewm_native_seed3072_e10'
LEGACY_DATA = Path('/content/lpwm-data')
if PHASE not in {'setup','published_eval','lewm_train','lpwm_train'}:
    raise ValueError('Nieznany PHASE: '+PHASE)

# Opcjonalny W&B: klucz pozostaje w środowisku, nie trafia do komend ani manifestów.
try:
    from google.colab import userdata
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    os.environ['WANDB_ENTITY'] = 'twojtys137-tw'
    os.environ['WANDB_PROJECT'] = 'lpwm-native'
    os.environ['WANDB_MODE'] = 'online'
except Exception:
    os.environ['WANDB_MODE'] = 'offline'
    print('Brak dostępnego sekretu W&B: logi offline + pliki na Drive.')

SCRIPT = REPO/'scripts/native_worldmodels.py'
def args_for(action, method='lewm', data=DATA):
    return [str(SCRIPT),action,'--method',method,'--work',str(WORK),
            '--data',str(data),'--output',str(OUTPUT)]
def native_python(method='lewm'):
    return str(WORK/method/'.venv/bin/python')
def run_setup(action, method='lewm', data=DATA):
    py = sys.executable if action in ('checkout','install') else native_python(method)
    subprocess.run([py,*args_for(action,method,data)],check=True)

def bounded(label, cap, command):
    global REMAINING_CU
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('Wybierz GPU A100 przed planowaniem i treningiem.')
    if REMAINING_CU is None or RATE_CU_HOUR is None:
        raise ValueError('Wpisz REMAINING_CU i RATE_CU_HOUR przed uruchomieniem GPU.')
    if cap > REMAINING_CU:
        raise ValueError('Limit tej komendy przekracza wpisany pozostały budżet.')
    ledger_path = OUTPUT/'budget.json'
    previous = json.loads(ledger_path.read_text()) if ledger_path.exists() else None
    campaign_budget = previous['total_cu'] if previous else REMAINING_CU
    def charged(record):
        return sum(r.get('estimated_cu',r['reserved_cu']) for r in record['runs']) if record else 0
    before = charged(previous)
    try:
        subprocess.run([sys.executable,'-m','experiments.budget','--ledger',str(ledger_path),
                        '--total-cu',str(campaign_budget),'--rate-cu-hour',str(RATE_CU_HOUR),
                        '--cap-cu',str(cap),'--label',label,'--',*command],check=True)
    finally:
        if ledger_path.exists():
            after = charged(json.loads(ledger_path.read_text()))
            REMAINING_CU -= max(0,after-before)
    print('Pozostały szacunek tej sesji:', REMAINING_CU)


## Przygotowanie natywnego LeWM

Oddzielne środowisko Pythona instaluje przypięte repo LeWM, zgodne API stable-worldmodel,
Transformers 4 i Pymunk 7. Oficjalne archiwum danych ma 13.1 GB; potrzebne jest też miejsce
na rozpakowany HDF 5. Pobranie można wznowić; dekompresja jest strumieniowa.
Kontrola checkpointu wymaga ścisłego dopasowania wszystkich wag.


In [ ]:
if PHASE in ('setup','published_eval','lewm_train'):
    run_setup('checkout')
    run_setup('install')
    run_setup('prepare-checkpoint')
    run_setup('prepare-data')
    print((OUTPUT/'lewm/data_audit.json').read_text())


## Opublikowany checkpoint: pierwsza kontrola planowania

Ta komórka uruchamia 10 zadań w natywnym protokole: cele 25 kroków dalej, budżet 50
surowych działań, CEM 300 × 30. Limit komendy to 60 szacowanych CU. Jeżeli wynik nadal
wynosi 0%, trening nie zostanie odblokowany — trzeba najpierw naprawić reprodukcję.
To mały test działania, nie końcowy wynik naukowy.


In [ ]:
if PHASE == 'published_eval':
    cmd = [sys.executable,*args_for('run'),'--task','published-eval','--n-evals','10',
           '--run-name',RUN_NAME,'--execute']
    bounded('lewm-published-eval10',60,cmd)
    print((OUTPUT/'lewm/published_eval_summary.json').read_text())


## LeWM: trening 10 epok i ewaluacja checkpointów

Ustaw `PHASE='lewm_train'` w pierwszej komórce po poprawnym teście checkpointu. Trening
korzysta z pełnych danych, batch 128, SIGReg 0.09/1024 i natywnych parametrów optymalizacji.
Maksymalny czas tego polecenia wynika z limitu 500 CU i wpisanej stawki. Nie wznawiamy
automatycznie istniejącego runu. Każda zakończona epoka zapisuje checkpoint.
Po zakończeniu oceniamy epoki 2,5,10 na tych samych 50 zadaniach.


In [ ]:
if PHASE == 'lewm_train':
    cmd = [sys.executable,*args_for('run'),'--task','train','--epochs','10',
           '--seed','3072','--run-name',RUN_NAME,'--execute']
    bounded('lewm-train10-'+RUN_NAME,500,cmd)
    for epoch in (2,5,10):
        cmd = [sys.executable,*args_for('run'),'--task','eval','--epoch',str(epoch),
               '--n-evals','50','--seed','3072','--run-name',RUN_NAME,'--execute']
        bounded(f'lewm-eval-e{epoch}-'+RUN_NAME,80,cmd)
else:
    subprocess.run([sys.executable,*args_for('run'),'--task','train',
                    '--epochs','10','--run-name',RUN_NAME],check=True)
    print('Powyżej jest plan komendy. Ustaw PHASE=lewm_train, aby wykonać trening.')


## Opcjonalnie: oficjalny LpWM, epoki 2/5/10

To repo Yiluna Kuanga, CLS 384/Deep-AdaLN/Reprelu/RDMReg 8192, batch 64, agg=b,
μP lr 1e-4, bez modyfikacji sparse-generator. Oficjalna recepta ma 2 epoki;
10 epok jest ablacją długości treningu. Korzysta z osobnego środowiska Pymunk 6.

Dane `pusht_noise` zostaną pobrane z oficjalnego OSF, jeżeli nie są dostępne pod LEGACY_DATA. Najpierw sprawdź ich **rzeczywistą**
liczbę trajektorii: `n_rollout=null` usuwa ograniczenie loadera, ale nie powiększa
pobranego archiwum. Nie porównuj automatycznie z LeWM: ten starszy protokół ma inne
zadania i większy maksymalny budżet działań. Ten etap wymaga dodatkowego zapasu CU.


In [ ]:
if PHASE == 'lpwm_train':
    run_setup('checkout','lpwm',LEGACY_DATA)
    run_setup('install','lpwm',LEGACY_DATA)
    run_setup('prepare-data','lpwm',LEGACY_DATA)
    print((OUTPUT/'lpwm/data_audit.json').read_text())
    name='lpwm_native_seed0_e10'
    cmd=[sys.executable,*args_for('run','lpwm',LEGACY_DATA),'--task','train',
         '--epochs','10','--seed','0','--run-name',name,'--execute']
    bounded('lpwm-train10-'+name,500,cmd)
    for epoch in (2,5,10):
        cmd=[sys.executable,*args_for('run','lpwm',LEGACY_DATA),'--task','eval',
             '--epoch',str(epoch),'--n-evals','50','--run-name',name,'--execute']
        bounded(f'lpwm-eval-e{epoch}-'+name,80,cmd)
else:
    print('LpWM pozostaje wyłączony. Wybierz PHASE=lpwm_train po sprawdzeniu danych i budżetu.')


Wyniki i manifesty są pod OUTPUT. `budget.json` zawiera szacunek zużycia wykonywanych
komend, a `environment/*-freeze.txt` zapis użytych zależności. Notebook nie twierdzi,
że jednostki lub success rate zostały już zmierzone.

DINO-WM pozostaje oddzielną reprodukcją: oficjalna konfiguracja używa 100 epok,
a publikacja i YAML różnią się learning rate predyktora. Szczegóły i przypięte
źródła: `docs/native_worldmodels.md`.
